<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/2_3_Linear_and_Quadratic_Discriminant_Analysis_(LDA_QDA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part II. Classification. Linear and Quadratic Discriminant Analysis (LDA/QDA)

## Глава 6. Линейный и квадратичный дискриминантный анализ (LDA/QDA)  


Метод k‑ближайших соседей не делал предположений о форме распределения признаков — он просто опирался на сходство объектов. Это дало ему гибкость: k‑NN способен выявлять границы любой сложности. Но за эту гибкость пришлось заплатить: алгоритм не выдаёт вероятности принадлежности к классам (оценка «доля соседей» крайне груба), требует хранения всей обучающей выборки, а при высокой размерности страдает от «проклятия размерности». Более того, у k‑NN нет интерпретируемой модели — мы не можем сказать, какие признаки важны и в какую сторону они влияют на решение.

Теперь мы переходим к **генеративному подходу** — принципиально иной философии классификации. Вместо того чтобы искать границу между классами напрямую, мы построим вероятностную модель того, как **признаки распределены внутри каждого класса**. Зная эти распределения, мы сможем по формуле Байеса вычислить вероятность того, что новый объект принадлежит каждому из классов, и выбрать наиболее вероятный.

Этот подход даёт нам сразу несколько преимуществ:
- **Хорошо калиброванные вероятности** — модель возвращает осмысленные апостериорные вероятности $P(Y=k \mid \mathbf{x})$, а не просто метку класса.
- **Компактность** — вместо хранения всех данных мы храним лишь небольшое число параметров (средние и ковариационные матрицы).
- **Интерпретируемость** — параметры модели имеют ясный геометрический и статистический смысл.
- **Визуализация** — LDA предоставляет мощный инструмент снижения размерности, который находит направления, максимально разделяющие классы.

В этой главе мы изучим два классических генеративных классификатора: **линейный дискриминантный анализ (LDA)** и **квадратичный дискриминантный анализ (QDA)**. Оба основаны на предположении о нормальности распределения признаков внутри классов, но LDA дополнительно предполагает, что все классы имеют одинаковую ковариационную матрицу — что приводит к **линейным** разделяющим границам, тогда как QDA допускает разные ковариации и даёт **квадратичные** границы.



### 1. Байесовский классификатор

Представьте, что вы — детектив, расследующий преступление. У вас есть улики (признаки) и несколько подозреваемых (классов). Вы знаете, насколько часто каждый подозреваемый вообще попадается в таких делах — это **априорная вероятность** $\pi_k = P(Y=k)$. Также вы знаете, насколько типичны улики для каждого из них — **функция правдоподобия** $p(\mathbf{x} \mid Y=k)$, то есть плотность распределения признаков $\mathbf{x}$ при условии, что объект принадлежит классу $k$. По формуле Байеса мы вычисляем **апостериорную вероятность** — степень уверенности в виновности подозреваемого после учёта улик:

$$
P(Y=k \mid \mathbf{x}) = \frac{\pi_k \, p(\mathbf{x} \mid Y=k)}{\sum_{j=1}^K \pi_j \, p(\mathbf{x} \mid Y=j)}.
$$

Классификатор, принимающий решение в пользу класса с максимальной апостериорной вероятностью, называется **байесовским классификатором** или правилом максимальной апостериорной вероятности (MAP):

$$
\hat{y} = \arg\max_k P(Y=k \mid \mathbf{x}).
$$

Такой классификатор теоретически оптимален: он минимизирует вероятность ошибки.

Поскольку знаменатель в формуле Байеса не зависит от $k$, мы можем сравнивать только числители, а ещё удобнее — их логарифмы. Определим **дискриминантные функции** как

$$
\delta_k(\mathbf{x}) = \log \bigl( \pi_k \, p(\mathbf{x} \mid Y=k) \bigr) = \log p(\mathbf{x} \mid Y=k) + \log \pi_k.
$$

Тогда решающее правило переписывается просто:

$$
\hat{y} = \arg\max_k \delta_k(\mathbf{x}).
$$

Граница между классами $i$ и $j$ задаётся уравнением $\delta_i(\mathbf{x}) = \delta_j(\mathbf{x})$. Переход к логарифмам превращает произведение вероятностей в сумму, а экспоненциальные выражения нормального распределения — в квадратичные формы, что резко упрощает анализ.

> **Углублённый взгляд: оптимальность байесовского классификатора**
>
> Пусть $R(\hat{y})$ — риск (ожидаемая ошибка) классификатора $\hat{y}$. Для любой решающей функции можно показать, что байесовский классификатор $\hat{y}^* = \arg\max_k P(Y=k \mid \mathbf{x})$ минимизирует вероятность неправильной классификации. В самом деле,
> $$
> P(\hat{y} \neq Y \mid \mathbf{x}) = 1 - P(Y = \hat{y} \mid \mathbf{x}),
> $$
> и выбор $\hat{y}$, максимизирующего апостериорную вероятность, очевидно, минимизирует это условное выражение. Интегрируя по $\mathbf{x}$, получаем минимальную полную ошибку. Таким образом, любой классификатор, стремящийся к байесовскому, автоматически движется к теоретическому пределу точности.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# Параметры двух нормальных классов
mu0 = np.array([0, 0])
mu1 = np.array([3, 2])
Sigma0 = np.array([[1.5, 0.5], [0.5, 1.0]])
Sigma1 = np.array([[1.2, -0.3], [-0.3, 1.5]])
pi0, pi1 = 0.6, 0.4

x = np.linspace(-4, 7, 300)
y = np.linspace(-4, 6, 300)
X, Y = np.meshgrid(x, y)
pos = np.dstack([X, Y])

# Плотности и апостериорная вероятность
p0 = multivariate_normal(mu0, Sigma0).pdf(pos)
p1 = multivariate_normal(mu1, Sigma1).pdf(pos)
post1 = pi1 * p1 / (pi0 * p0 + pi1 * p1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].contourf(X, Y, p0, cmap='Blues', alpha=0.6)
axes[0].contour(X, Y, p0, colors='navy', linewidths=0.5)
axes[0].set_title(r'$p(\mathbf{x} \mid Y=0)$')
axes[1].contourf(X, Y, p1, cmap='Reds', alpha=0.6)
axes[1].contour(X, Y, p1, colors='darkred', linewidths=0.5)
axes[1].set_title(r'$p(\mathbf{x} \mid Y=1)$')
axes[2].contourf(X, Y, post1, cmap='coolwarm', alpha=0.6,
                 levels=np.linspace(0, 1, 11))
axes[2].contour(X, Y, post1, levels=[0.5], colors='black', linewidths=2)
axes[2].set_title(r'$P(Y=1 \mid \mathbf{x})$ и граница')
for ax in axes:
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()



### 2. Многомерное нормальное распределение (краткое напоминание)

Байесовский классификатор требует знания $p(\mathbf{x} \mid Y=k)$. LDA и QDA предполагают, что признаки в каждом классе распределены по многомерному нормальному закону. Его плотность:

$$
p(\mathbf{x}) = \frac{1}{(2\pi)^{d/2} |\Sigma|^{1/2}} \exp\!\left( -\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^\mathsf{T} \Sigma^{-1} (\mathbf{x} - \boldsymbol{\mu}) \right).
$$

Здесь $\boldsymbol{\mu} \in \mathbb{R}^d$ — вектор математических ожиданий (центр облака), а $\Sigma \in \mathbb{R}^{d \times d}$ — ковариационная матрица, симметричная и положительно определённая. Геометрически линии постоянной плотности — это эллипсоиды, центрированные в $\boldsymbol{\mu}$. Их оси направлены вдоль собственных векторов $\Sigma$, а длины пропорциональны квадратным корням из собственных значений.

Ключевая величина — **расстояние Махаланобиса**:

$$
\Delta^2(\mathbf{x}, \boldsymbol{\mu}) = (\mathbf{x} - \boldsymbol{\mu})^\mathsf{T} \Sigma^{-1} (\mathbf{x} - \boldsymbol{\mu}),
$$

которое измеряет удалённость точки от центра с учётом корреляций и масштаба признаков: в направлении большой дисперсии расстояние сжимается, компенсируя естественный разброс.

---

### 3. Квадратичный дискриминантный анализ (QDA)

Предположим, что каждый класс $k$ имеет своё нормальное распределение:

$$
p(\mathbf{x} \mid Y=k) = \mathcal{N}(\boldsymbol{\mu}_k, \Sigma_k).
$$

Подставим эту плотность в дискриминантную функцию $\delta_k(\mathbf{x}) = \log p(\mathbf{x} \mid Y=k) + \log \pi_k$, опуская члены, не зависящие от $k$:

$$
\delta_k(\mathbf{x}) = -\frac{1}{2} \log |\Sigma_k| - \frac{1}{2}(\mathbf{x} - \boldsymbol{\mu}_k)^\mathsf{T} \Sigma_k^{-1} (\mathbf{x} - \boldsymbol{\mu}_k) + \log \pi_k.
\tag{1}
$$

Это — квадратичная функция от $\mathbf{x}$, поскольку $(\mathbf{x} - \boldsymbol{\mu}_k)^\mathsf{T} \Sigma_k^{-1} (\mathbf{x} - \boldsymbol{\mu}_k)$ раскрывается в $\mathbf{x}^\mathsf{T} \Sigma_k^{-1} \mathbf{x} - 2\boldsymbol{\mu}_k^\mathsf{T} \Sigma_k^{-1} \mathbf{x} + \boldsymbol{\mu}_k^\mathsf{T} \Sigma_k^{-1} \boldsymbol{\mu}_k$. Поэтому граница между классами $i$ и $j$, $\delta_i(\mathbf{x}) = \delta_j(\mathbf{x})$, принимает вид

$$
-\frac{1}{2}\mathbf{x}^\mathsf{T} (\Sigma_i^{-1} - \Sigma_j^{-1}) \mathbf{x} + \dots = 0,
$$

где многоточие обозначает линейные и константные члены. Это уравнение квадрики — в общем случае оно может описывать эллипсоиды, гиперболоиды, параболоиды или даже пары пересекающихся плоскостей.

Число параметров QDA для $K$ классов и $d$ признаков:
- векторов средних: $K \cdot d$;
- уникальных элементов ковариационных матриц: $K \cdot \frac{d(d+1)}{2}$.

Итого $K\left(d + \frac{d(d+1)}{2}\right)$ параметров. При малых выборках, особенно если $n_k \le d$, оценка $\Sigma_k$ может оказаться вырожденной, и QDA теряет устойчивость.



In [ ]:
np.random.seed(42)
n = 150
# Класс 0 — вытянут по диагонали
X0 = np.random.multivariate_normal([-1, -1], [[3, 2.5], [2.5, 3]], n)
# Класс 1 — сферический
X1 = np.random.multivariate_normal([2, 1], [[0.8, 0], [0, 0.8]], n)

X_syn = np.vstack([X0, X1])
y_syn = np.array([0]*n + [1]*n)

from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)

lda_syn = LinearDiscriminantAnalysis().fit(X_syn, y_syn)
qda_syn = QuadraticDiscriminantAnalysis().fit(X_syn, y_syn)

xx, yy = np.meshgrid(np.linspace(-6, 7, 300), np.linspace(-5, 6, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
Z_lda = lda_syn.predict(grid).reshape(xx.shape)
Z_qda = qda_syn.predict(grid).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, Z, title in zip(axes, [Z_lda, Z_qda], ['LDA', 'QDA']):
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X0[:,0], X0[:,1], c='blue', edgecolor='k', s=30, label='Класс 0')
    ax.scatter(X1[:,0], X1[:,1], c='red', edgecolor='k', s=30, label='Класс 1')
    ax.set_title(title)
    ax.legend(); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()


### 4. Линейный дискриминантный анализ (LDA)

LDA добавляет ключевое предположение: **все классы имеют одинаковую ковариационную матрицу** $\Sigma_1 = \dots = \Sigma_K = \Sigma$. Тогда дискриминантная функция (1) упрощается, потому что квадратичный член $\mathbf{x}^\mathsf{T} \Sigma^{-1} \mathbf{x}$ не зависит от $k$ и сокращается при сравнении. Остаётся

$$
\delta_k(\mathbf{x}) = \mathbf{x}^\mathsf{T} \Sigma^{-1} \boldsymbol{\mu}_k - \frac{1}{2} \boldsymbol{\mu}_k^\mathsf{T} \Sigma^{-1} \boldsymbol{\mu}_k + \log \pi_k.
\tag{2}
$$

Это **линейная** по $\mathbf{x}$ функция. Граница между классами $i$ и $j$ становится гиперплоскостью:

$$
\mathbf{x}^\mathsf{T} \Sigma^{-1} (\boldsymbol{\mu}_i - \boldsymbol{\mu}_j) = \frac{1}{2}\left( \boldsymbol{\mu}_i^\mathsf{T} \Sigma^{-1} \boldsymbol{\mu}_i - \boldsymbol{\mu}_j^\mathsf{T} \Sigma^{-1} \boldsymbol{\mu}_j \right) - \log\frac{\pi_i}{\pi_j}.
$$

Если классы равновероятны ($\pi_i = \pi_j$), граница проходит через середину отрезка между центрами $\boldsymbol{\mu}_i$ и $\boldsymbol{\mu}_j$ в смысле расстояния Махаланобиса и перпендикулярна вектору $\Sigma^{-1}(\boldsymbol{\mu}_i - \boldsymbol{\mu}_j)$.

Число параметров LDA:
- векторов средних: $K \cdot d$;
- элементов общей ковариационной матрицы: $\frac{d(d+1)}{2}$.

Это существенно меньше, чем у QDA, что делает LDA более устойчивым, особенно при малых обучающих выборках.

> **Углублённый взгляд: связь LDA с линейной регрессией**
>
> В случае бинарной классификации ($y \in \{0,1\}$) и сбалансированных классов ($\pi_0 = \pi_1$) LDA даёт коэффициенты, пропорциональные коэффициентам линейной регрессии, предсказывающей индикатор класса. Действительно, линейная регрессия минимизирует $\sum (y_i - \beta_0 - \mathbf{x}_i^\mathsf{T}\boldsymbol{\beta})^2$. Можно показать, что её решение $\hat{\boldsymbol{\beta}}_{\text{LS}}$ пропорционально $\Sigma^{-1}(\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0)$ — тому же направлению, которое определяет LDA-границу. Точнее, если $\hat{\boldsymbol{\beta}}_{\text{LS}}$ — оценка МНК, то $\hat{\boldsymbol{\beta}}_{\text{LDA}} \propto \Sigma^{-1}(\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0) \propto \hat{\boldsymbol{\beta}}_{\text{LS}}$ при равных размерах классов. Это объясняет, почему LDA можно рассматривать как линейную модель, обученную через вероятностную порождающую спецификацию, а линейная регрессия даёт дискриминантное правило, совпадающее с LDA по направлению.





### 5. Оценка параметров по выборке

На практике истинные параметры распределений неизвестны, и их заменяют выборочными оценками, полученными по обучающей выборке $\{(\mathbf{x}^{(i)}, y_i)\}_{i=1}^n$. Все оценки ниже выводятся методом максимального правдоподобия в предположении нормальности, с поправками на несмещённость там, где это необходимо.

**Априорные вероятности** классов оцениваются долями объектов соответствующего класса в обучающей выборке:

$$
\hat{\pi}_k = \frac{n_k}{n}, \qquad k = 1,\dots,K,
$$

где $n_k = |\{i : y_i = k\}|$ — число объектов класса $k$. Эта оценка является оценкой максимального правдоподобия для мультиномиального распределения меток.

**Векторы средних** для каждого класса вычисляются как выборочные средние:

$$
\hat{\boldsymbol{\mu}}_k = \frac{1}{n_k} \sum_{i: y_i=k} \mathbf{x}^{(i)}. \tag{5.1}
$$

Это несмещённая оценка истинного вектора математического ожидания $\boldsymbol{\mu}_k$, что следует из линейности математического ожидания: $\mathbb{E}[\hat{\boldsymbol{\mu}}_k] = \frac{1}{n_k} \sum_{i: y_i=k} \mathbb{E}[\mathbf{x}^{(i)}] = \boldsymbol{\mu}_k$.

**Общая ковариационная матрица LDA (pooled covariance).** В предположении равенства ковариационных матриц всех классов $\Sigma_1 = \cdots = \Sigma_K = \Sigma$ используется сводная оценка, агрегирующая информацию по всем классам:

$$
\hat{\Sigma} = \frac{1}{n-K} \sum_{k=1}^K \sum_{i: y_i=k} (\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)^\mathsf{T}. \tag{5.2}
$$

**Почему в знаменателе $n-K$, а не $n$?** Оценка максимального правдоподобия использовала бы знаменатель $n$, однако такая оценка смещена: она систематически занижает истинную ковариационную матрицу. Причина в том, что при центрировании на выборочное среднее $\hat{\boldsymbol{\mu}}_k$ вместо истинного $\boldsymbol{\mu}_k$ мы теряем по одной степени свободы на каждый класс. Формально, для любого класса $k$ справедливо:

$$
\mathbb{E}\left[ \sum_{i: y_i=k} (\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)^\mathsf{T} \right] = (n_k - 1)\,\Sigma.
$$

Покажем это. Пусть $\mathbf{z}_i = \mathbf{x}^{(i)} - \boldsymbol{\mu}_k$ — центрированные наблюдения с нулевым средним и ковариационной матрицей $\Sigma$. Тогда $\hat{\boldsymbol{\mu}}_k - \boldsymbol{\mu}_k = \frac{1}{n_k} \sum_{i=1}^{n_k} \mathbf{z}_i$. Раскладывая сумму квадратов,

$$
\begin{aligned}
\sum_{i: y_i=k} (\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)^\mathsf{T}
&= \sum_{i=1}^{n_k} \mathbf{z}_i \mathbf{z}_i^\mathsf{T} - n_k (\hat{\boldsymbol{\mu}}_k - \boldsymbol{\mu}_k)(\hat{\boldsymbol{\mu}}_k - \boldsymbol{\mu}_k)^\mathsf{T}.
\end{aligned}
$$

Математическое ожидание первого слагаемого равно $n_k \Sigma$. Математическое ожидание второго слагаемого:

$$
n_k \,\mathbb{E}\left[ \left(\frac{1}{n_k}\sum_i \mathbf{z}_i\right) \left(\frac{1}{n_k}\sum_j \mathbf{z}_j\right)^\mathsf{T} \right] = \frac{1}{n_k} \sum_{i,j} \mathbb{E}[\mathbf{z}_i \mathbf{z}_j^\mathsf{T}] = \frac{1}{n_k} \cdot n_k \Sigma = \Sigma,
$$

поскольку $\mathbb{E}[\mathbf{z}_i \mathbf{z}_j^\mathsf{T}] = 0$ при $i \neq j$ в силу независимости наблюдений. Итого:

$$
\mathbb{E}\left[ \sum_{i: y_i=k} (\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)^\mathsf{T} \right] = n_k \Sigma - \Sigma = (n_k - 1)\Sigma.
$$

Суммируя по всем $K$ классам, получаем:

$$
\mathbb{E}\left[ \sum_{k=1}^K \sum_{i: y_i=k} (\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)^\mathsf{T} \right] = \sum_{k=1}^K (n_k - 1)\Sigma = (n - K)\Sigma,
$$

откуда непосредственно следует, что оценка (5.2) с делителем $n-K$ является несмещённой: $\mathbb{E}[\hat{\Sigma}] = \Sigma$.

**Индивидуальные ковариационные матрицы QDA.** Когда предположение об общей ковариации не вводится, для каждого класса оценивается своя матрица:

$$
\hat{\Sigma}_k = \frac{1}{n_k - 1} \sum_{i: y_i=k} (\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}^{(i)} - \hat{\boldsymbol{\mu}}_k)^\mathsf{T}. \tag{5.3}
$$

Делитель $n_k - 1$ обеспечивает несмещённость по той же причине, что и в (5.2), но уже для одного класса.

**Проблема вырожденности.** Если для некоторого класса число наблюдений не превосходит размерности признакового пространства ($n_k \le d$), матрица $\hat{\Sigma}_k$ оказывается вырожденной: её ранг не превышает $n_k - 1 < d$, и обратная матрица $\hat{\Sigma}_k^{-1}$, необходимая для вычисления дискриминантной функции (1), не существует. В этом случае QDA неприменим без регуляризации. Простейший способ регуляризации — добавить к выборочной ковариационной матрице малую диагональную подмесь: $\hat{\Sigma}_k^{\text{reg}} = \hat{\Sigma}_k + \lambda I$, где $\lambda > 0$ — параметр регуляризации. Это гарантирует положительную определённость и обратимость ценой небольшого смещения оценки.

LDA лишён проблемы вырожденности на уровне отдельных классов, поскольку pooled-матрица (5.2) агрегирует данные всех классов, и её ранг может достигать $\min(d, n-K)$. Однако при $d$, сравнимом с $n$, оценка $\hat{\Sigma}$ становится плохо обусловленной: её собственные значения, соответствующие направлениям с малой дисперсией, оцениваются с большой ошибкой, что приводит к неустойчивости обратной матрицы. В таких случаях применяют **shrinkage-оценку** (Ledoit–Wolf или эквивалентную регуляризацию Тихонова):

$$
\hat{\Sigma}_{\alpha} = (1-\alpha)\hat{\Sigma} + \alpha \frac{\text{tr}(\hat{\Sigma})}{d} I,
$$

где $\alpha \in [0,1]$ подбирается аналитически или кросс‑валидацией (см. Часть 2, раздел 2). Это смещает оценку в сторону сферической матрицы, но резко уменьшает её дисперсию и улучшает обусловленность.




### 6. Сравнение LDA и QDA: компромисс смещения и дисперсии

Различие между LDA и QDA — это классический пример фундаментального компромисса между смещением и дисперсией (bias‑variance tradeoff), проявляющегося на уровне спецификации модели.

**LDA** жёстко предполагает равенство ковариационных матриц всех классов: $\Sigma_1 = \dots = \Sigma_K = \Sigma$. Это предположение вносит **смещение** — если истинные ковариационные матрицы классов различны, модель LDA принципиально не способна воспроизвести истинную форму разделяющих границ. Однако это же предположение резко сокращает число свободных параметров:

- LDA оценивает $K \cdot d$ средних и $\frac{d(d+1)}{2}$ элементов общей ковариационной матрицы — итого $O(Kd + d^2)$ параметров.
- QDA оценивает $K \cdot d$ средних и $K \cdot \frac{d(d+1)}{2}$ элементов индивидуальных ковариационных матриц — итого $O(Kd + Kd^2)$ параметров.

При фиксированном объёме выборки $n$ меньшее число параметров означает, что каждый параметр LDA оценивается по большему числу наблюдений, что ведёт к меньшей **дисперсии** оценок. Грубо говоря, LDA «усредняет» информацию о ковариационной структуре по всем классам, и если предположение о равенстве матриц не слишком нарушено, это усреднение идёт на пользу.

**QDA**, напротив, свободен от предположения об одинаковых ковариациях. Он способен воспроизводить границы практически любой формы (эллипсы, гиперболы, параболы), что даёт ему низкое **смещение**. Однако за гибкость приходится платить: число параметров растёт пропорционально числу классов $K$, и для каждого класса требуется оценить свою матрицу $\Sigma_k$. Если $n_k$ (число наблюдений в классе $k$) невелико по сравнению с $d$, оценка $\hat{\Sigma}_k$ становится крайне неустойчивой — её дисперсия велика, и, как следствие, велика дисперсия предсказаний. В предельном случае $n_k \le d$ матрица вообще вырождается, и QDA неприменим без регуляризации.

**Формальная иллюстрация.** В асимптотике, когда $n \to \infty$ и $d$ фиксировано, QDA всегда не хуже LDA, поскольку способно сколь угодно точно оценить истинные ковариационные матрицы. Однако при конечных $n$ ситуация иная. Ожидаемую дополнительную ошибку QDA по сравнению с LDA можно приближённо разложить как

$$
\Delta \text{Err} \approx \underbrace{\text{снижение смещения}}_{\text{в пользу QDA}} \;-\; \underbrace{\frac{d(d+1)}{2} \sum_{k=1}^K \left( \frac{1}{n_k} - \frac{1}{n} \right) \cdot c(\Sigma_k, \Sigma)}_{\text{рост дисперсии}}.
$$

Даже без точного вывода этого выражения видно, что с ростом $d$ и уменьшением $n_k$ дисперсионный штраф QDA быстро растёт, сводя на нет выигрыш от уменьшения смещения.

**Визуализация компромисса.** Приведённый ниже код моделирует два класса с заведомо разными ковариационными матрицами и варьирует размер обучающей выборки. Построим кривые тестовой ошибки для LDA и QDA.




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.model_selection import train_test_split

np.random.seed(42)
d = 5
mu0 = np.zeros(d)
mu1 = np.ones(d) * 1.5
Sigma0 = np.eye(d)
Sigma1 = np.eye(d) * 3 + 0.5 * np.ones((d, d))  # класс 1 более разбросан

train_sizes = [20, 30, 50, 80, 120, 200, 400]
lda_errs, qda_errs = [], []

for n_train in train_sizes:
    lda_acc, qda_acc = [], []
    for _ in range(100):
        X0 = np.random.multivariate_normal(mu0, Sigma0, n_train // 2)
        X1 = np.random.multivariate_normal(mu1, Sigma1, n_train // 2)
        X = np.vstack([X0, X1])
        y = np.array([0]*(n_train//2) + [1]*(n_train//2))
        X_test0 = np.random.multivariate_normal(mu0, Sigma0, 500)
        X_test1 = np.random.multivariate_normal(mu1, Sigma1, 500)
        X_test = np.vstack([X_test0, X_test1])
        y_test = np.array([0]*500 + [1]*500)

        lda = LinearDiscriminantAnalysis().fit(X, y)
        qda = QuadraticDiscriminantAnalysis().fit(X, y)
        lda_acc.append(np.mean(lda.predict(X_test) == y_test))
        qda_acc.append(np.mean(qda.predict(X_test) == y_test))

    lda_errs.append(1 - np.mean(lda_acc))
    qda_errs.append(1 - np.mean(qda_acc))

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, lda_errs, 'o-', label='LDA', linewidth=2)
plt.plot(train_sizes, qda_errs, 's--', label='QDA', linewidth=2)
plt.xlabel('Размер обучающей выборки $n$')
plt.ylabel('Ошибка на тесте')
plt.title('LDA vs QDA: компромисс смещения и дисперсии ($d=5$)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



**Интерпретация.** При малых $n$ LDA выигрывает за счёт меньшей дисперсии, несмотря на смещённость предположения об общей ковариации. С ростом выборки оценки QDA стабилизируются, его смещение уменьшается, и QDA выходит вперёд — именно так выглядит классический bias‑variance tradeoff.

**Практическое правило:**
- Если $n_k \gg d$ для всех классов, пробуйте QDA — он может выиграть за счёт лучшей подгонки границы.
- Если данных мало или $d$ сравнимо с $n_k$, начинайте с LDA как с простого, устойчивого и интерпретируемого baseline.
- В промежуточных ситуациях используйте кросс‑валидацию для прямого сравнения LDA и QDA (а также регуляризованного LDA со shrinkage) на ваших данных.




### Вопросы для самопроверки

#### Для начинающих
1. Что максимизирует байесовский классификатор? Почему он называется оптимальным?
2. Запишите дискриминантную функцию LDA и объясните, почему она линейна.
3. Чем геометрически граница QDA отличается от границы LDA? Приведите примеры форм.
4. Что такое расстояние Махаланобиса и почему оно важно для LDA/QDA?
5. Почему в LDA меньше параметров, чем в QDA? К чему это приводит на практике?
6. Как оценивается общая ковариационная матрица в LDA? Почему используется знаменатель $n-K$?
7. Как интерпретировать коэффициенты дискриминантной функции LDA? Можно ли сказать, что они указывают «важность» признаков?

#### Для профессионалов
1. Выведите явное уравнение разделяющей гиперплоскости в LDA для двух классов с равными априорными вероятностями.
2. Докажите эквивалентность направления LDA и коэффициентов линейной регрессии для бинарной классификации (при равных размерах классов).
3. Как собственные векторы матрицы $\Sigma^{-1}$ связаны с геометрией LDA-границ?
4. Что происходит с QDA, если для некоторого класса $n_k \le d$? Какие решения предлагаются?
5. Пусть истинные ковариационные матрицы классов различны. Сравните ожидаемую байесовскую ошибку LDA и QDA: в каком случае LDA проигрывает сильнее и почему?



## Часть 2. Реализация и применение LDA/QDA на Python

В первой части мы заложили теоретический фундамент: байесовский классификатор, предположение о нормальности и вытекающие из него линейные и квадратичные дискриминантные функции. Теперь перейдём к практике — реализуем алгоритмы с нуля, научимся пользоваться готовыми инструментами `sklearn`, оценим качество моделей и проверим, насколько данные соответствуют нашим предположениям. Начнём с единого блока импортов.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, chi2
import scipy.linalg as la

from sklearn.datasets import load_wine, load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, auc,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.dpi'] = 120
sns.set_style("whitegrid")

### 1. Реализация LDA и QDA с нуля

Построим два класса — `LDA` и `QDA`, следуя формулам из первой части. Метод `fit` оценивает параметры распределений, `predict` применяет MAP-правило, а `predict_proba` возвращает апостериорные вероятности по формуле Байеса. Для численной устойчивости в QDA добавим небольшую диагональную подмесь к ковариационным матрицам — простейшую регуляризацию.



In [ ]:
class LDA:
    def __init__(self):
        self.classes_ = None
        self.priors_ = {}
        self.means_ = {}
        self.Sigma_ = None
        self.invSigma_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n, d = X.shape
        K = len(self.classes_)
        pooled_cov = np.zeros((d, d))
        for k in self.classes_:
            k = int(k)
            Xk = X[y == k]
            nk = len(Xk)
            self.priors_[k] = nk / n
            self.means_[k] = np.mean(Xk, axis=0)
            centered = Xk - self.means_[k]
            pooled_cov += centered.T @ centered
        self.Sigma_ = pooled_cov / (n - K)
        self.invSigma_ = la.inv(self.Sigma_)
        return self

    def predict_proba(self, X):
        K = len(self.classes_)
        proba = np.zeros((X.shape[0], K))
        for idx, k in enumerate(self.classes_):
            k = int(k)
            diff = X - self.means_[k]
            log_prob = -0.5 * np.sum(diff @ self.invSigma_ * diff, axis=1) + np.log(self.priors_[k])
            proba[:, idx] = log_prob
        proba -= np.max(proba, axis=1, keepdims=True)
        proba = np.exp(proba)
        proba /= np.sum(proba, axis=1, keepdims=True)
        return proba

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


class QDA:
    def __init__(self, reg=1e-6):
        self.reg = reg
        self.classes_ = None
        self.priors_ = {}
        self.means_ = {}
        self.Sigmas_ = {}
        self.invSigmas_ = {}
        self.logdets_ = {}

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n, d = X.shape
        for k in self.classes_:
            k = int(k)
            Xk = X[y == k]
            nk = len(Xk)
            self.priors_[k] = nk / n
            self.means_[k] = np.mean(Xk, axis=0)
            centered = Xk - self.means_[k]
            cov = centered.T @ centered / (nk - 1) if nk > 1 else np.eye(d)
            cov += self.reg * np.eye(d)
            self.Sigmas_[k] = cov
            self.invSigmas_[k] = la.inv(cov)
            sign, logdet = np.linalg.slogdet(cov)
            self.logdets_[k] = logdet
        return self

    def predict_proba(self, X):
        K = len(self.classes_)
        proba = np.zeros((X.shape[0], K))
        for idx, k in enumerate(self.classes_):
            k = int(k)
            diff = X - self.means_[k]
            log_prob = (-0.5 * np.sum(diff @ self.invSigmas_[k] * diff, axis=1)
                        - 0.5 * self.logdets_[k] + np.log(self.priors_[k]))
            proba[:, idx] = log_prob
        proba -= np.max(proba, axis=1, keepdims=True)
        proba = np.exp(proba)
        proba /= np.sum(proba, axis=1, keepdims=True)
        return proba

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

Протестируем на синтетическом примере с тремя нормальными облаками, имеющими разные ковариационные матрицы, — классический сценарий, где QDA должен давать выигрыш.

In [ ]:
np.random.seed(123)
n_samples = 200
d = 2

# класс 0 — вытянут вдоль главной диагонали
mean0 = np.array([-2, -2])
cov0 = np.array([[3, 2.5], [2.5, 3]])
# класс 1 — сферический
mean1 = np.array([1, -1])
cov1 = np.array([[1, 0], [0, 1]])
# класс 2 — вытянут вдоль оси x
mean2 = np.array([-1, 2])
cov2 = np.array([[2, 0], [0, 0.5]])

X0 = np.random.multivariate_normal(mean0, cov0, n_samples)
X1 = np.random.multivariate_normal(mean1, cov1, n_samples)
X2 = np.random.multivariate_normal(mean2, cov2, n_samples)

X_syn = np.vstack([X0, X1, X2])
y_syn = np.array([0]*n_samples + [1]*n_samples + [2]*n_samples)

# Обучение моделей
lda_syn = LDA().fit(X_syn, y_syn)
qda_syn = QDA().fit(X_syn, y_syn)

# Построение контуров апостериорных вероятностей и границ решений
xx, yy = np.meshgrid(np.linspace(X_syn[:,0].min()-1, X_syn[:,0].max()+1, 300),
                     np.linspace(X_syn[:,1].min()-1, X_syn[:,1].max()+1, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, name in zip(axes,
                           [lda_syn, qda_syn],
                           ['LDA (с нуля)', 'QDA (с нуля)']):
    proba = model.predict_proba(grid)
    # Отображаем цветом класс с максимальной вероятностью
    pred = model.classes_[np.argmax(proba, axis=1)].reshape(xx.shape)
    ax.contourf(xx, yy, pred, alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5],
                colors=['#ff9999','#99ccff','#99ff99'])
    ax.scatter(X_syn[:,0], X_syn[:,1], c=y_syn, edgecolor='k', cmap='viridis', s=30)
    ax.set_title(name)
    ax.set_xlabel('Признак 1'); ax.set_ylabel('Признак 2')
plt.tight_layout()
plt.show()



*[Рисунок 1: границы решений LDA и QDA, построенных с нуля, на трёхклассовых данных с разными ковариациями.]*

LDA даёт линейные границы, QDA — криволинейные, лучше повторяющие истинную форму облаков. В области пересечения классов QDA строит более гибкую границу.

> **Углублённый взгляд: теоретические границы для QDA**
>
> Если известны истинные параметры $\boldsymbol{\mu}_k$ и $\Sigma_k$, можно построить байесовскую границу — поверхность, на которой апостериорные вероятности двух классов равны. Для нормальных распределений это квадрика. Сравнение с ней показывает, насколько качественно QDA восстанавливает истинную картину при конечной выборке. В нашем коде мы не выводим теоретическую границу, но при желании её можно добавить, зная параметры генерации.

---

### 2. Использование sklearn: LDA, QDA и shrinkage

Теперь обратимся к `scikit-learn`, где LDA и QDA уже реализованы. Возьмём набор данных `wine` — химический анализ вин трёх сортов — и сравним обычный LDA, QDA и регуляризованный LDA со shrinkage.



In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis()

lda.fit(X_train_s, y_train)
qda.fit(X_train_s, y_train)

y_pred_lda = lda.predict(X_test_s)
y_pred_qda = qda.predict(X_test_s)

print("LDA accuracy:", accuracy_score(y_test, y_pred_lda))
print("QDA accuracy:", accuracy_score(y_test, y_pred_qda))
print("\nLDA classification report:\n", classification_report(y_test, y_pred_lda))
print("QDA classification report:\n", classification_report(y_test, y_pred_qda))

Построим матрицы ошибок и визуализируем границы решений на паре признаков.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# Матрицы ошибок
for ax, y_pred, title in zip(axes[:2],
                             [y_pred_lda, y_pred_qda],
                             ['LDA', 'QDA']):
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax,
                                            cmap='Blues', colorbar=False)
    ax.set_title(title)

# Границы на первых двух признаках
ax = axes[2]
X2_train = X_train_s[:, :2]
X2_test = X_test_s[:, :2]
lda2 = LinearDiscriminantAnalysis().fit(X2_train, y_train)
qda2 = QuadraticDiscriminantAnalysis().fit(X2_train, y_train)

xx, yy = np.meshgrid(np.linspace(X2_test[:,0].min()-1, X2_test[:,0].max()+1, 200),
                     np.linspace(X2_test[:,1].min()-1, X2_test[:,1].max()+1, 200))
grid2 = np.c_[xx.ravel(), yy.ravel()]
Z_lda = lda2.predict(grid2).reshape(xx.shape)
Z_qda = qda2.predict(grid2).reshape(xx.shape)

# Покажем обе границы на одном графике (контуры LDA и заливка QDA)
ax.contourf(xx, yy, Z_qda, alpha=0.2, levels=[-0.5, 0.5, 1.5, 2.5],
            colors=['#ffaaaa','#aaffaa','#aaaaff'])
ax.contour(xx, yy, Z_lda, levels=[-0.5, 0.5, 1.5, 2.5],
           colors='black', linewidths=2, linestyles='dashed')
ax.scatter(X2_test[:,0], X2_test[:,1], c=y_test, edgecolor='k', cmap='viridis')
ax.set_title('Границы LDA (пунктир) и QDA (фон)')
ax.set_xlabel(wine.feature_names[0])
ax.set_ylabel(wine.feature_names[1])
plt.tight_layout()
plt.show()


*[Рисунок 2: матрицы ошибок LDA и QDA, а также сравнение границ на первых двух признаках вина.]*

QDA строит слегка искривлённую границу, в то время как LDA довольствуется прямой.

**Регуляризация LDA (shrinkage).** Идея проста: заменить выборочную ковариационную матрицу $\hat{\Sigma}$ на сжатую версию $\hat{\Sigma}_{\alpha} = (1-\alpha)\hat{\Sigma} + \alpha \frac{\text{tr}(\hat{\Sigma})}{d} I$, где $\alpha \in [0,1]$ — параметр shrinkage. Это уменьшает дисперсию оценки и повышает устойчивость, особенно когда число признаков $d$ близко к числу наблюдений $n$. В `sklearn` можно включить автоматический выбор $\alpha$ через `solver='lsqr'` и `shrinkage='auto'` (Ledoit‑Wolf). Сравним на искусственно созданной задаче с высокой корреляцией признаков и малым $n$.



In [ ]:
# Генерируем данные с сильной корреляцией признаков
np.random.seed(0)
n_features = 30
n_train = 60
X_corr, y_corr = make_classification(n_samples=200, n_features=n_features,
                                     n_informative=15, random_state=42)
X_corr_train, X_corr_test, y_corr_train, y_corr_test = train_test_split(
    X_corr, y_corr, train_size=n_train, random_state=42)

lda_vanilla = LinearDiscriminantAnalysis(solver='svd')
lda_shrink = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')

lda_vanilla.fit(X_corr_train, y_corr_train)
lda_shrink.fit(X_corr_train, y_corr_train)

acc_vanilla = accuracy_score(y_corr_test, lda_vanilla.predict(X_corr_test))
acc_shrink = accuracy_score(y_corr_test, lda_shrink.predict(X_corr_test))

print(f"LDA без shrinkage: {acc_vanilla:.4f}")
print(f"LDA со shrinkage (Ledoit‑Wolf): {acc_shrink:.4f}")



Регуляризованный LDA часто заметно выигрывает, когда $n$ невелико, а признаки сильно коррелированы, — классический компромисс смещения и дисперсии на уровне оценки ковариационной матрицы.

> **Углублённый взгляд: оптимальный shrinkage**
>
> Ledoit и Wolf (2004) предложили аналитическую формулу для оптимального $\alpha$, минимизирующего ожидаемую квадратичную ошибку оценки ковариационной матрицы относительно истинной. Этот подход не требует кросс‑валидации и реализован в `sklearn`. Альтернативно можно перебирать $\alpha$ вручную, но автоматический вариант даёт близкие результаты.

---

### 3. Метрики качества и ROC‑анализ

Метрики для многоклассовой классификации (accuracy, macro‑F1, precision/recall по классам) мы уже применили. Теперь сосредоточимся на бинарном случае и сравним вероятностные выходы LDA/QDA с логистической регрессией.

Преобразуем задачу `wine` в бинарную: отделим класс 0 от классов 1 и 2.



In [ ]:
y_bin = (y != 0).astype(int)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X, y_bin,
                                                            test_size=0.3, random_state=42)
scaler_bin = StandardScaler().fit(X_train_b)
X_train_b_s = scaler_bin.transform(X_train_b)
X_test_b_s = scaler_bin.transform(X_test_b)

lda_bin = LinearDiscriminantAnalysis()
qda_bin = QuadraticDiscriminantAnalysis()
logreg = LogisticRegression(max_iter=5000)

for model, name in zip([lda_bin, qda_bin, logreg],
                       ['LDA', 'QDA', 'Logistic Regression']):
    model.fit(X_train_b_s, y_train_b)
    y_prob = model.predict_proba(X_test_b_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_b, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0,1],[0,1], 'k--', alpha=0.3)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC‑кривые для бинарной классификации вина')
plt.legend()
plt.tight_layout()
plt.show()



*[Рисунок 3: ROC‑кривые LDA, QDA и логистической регрессии.]*

Генеративные классификаторы LDA и QDA дают хорошо откалиброванные вероятности, когда предположение о нормальности выполняется, и их AUC может быть не ниже, чем у логистической регрессии. На малых выборках «дополнительная структура» в виде нормальной модели часто делает LDA/QDA более эффективными, чем чисто дискриминативная логистическая регрессия. Когда же нормальность сильно нарушена, логистическая регрессия, не полагающаяся на это предположение, может оказаться точнее.

> **Углублённый взгляд: генеративный vs дискриминативный подход**
>
> LDA моделирует совместное распределение $p(\mathbf{x}, y)$, тогда как логистическая регрессия — напрямую $p(y \mid \mathbf{x})$. Если предположение о нормальности верно, LDA достигает асимптотической эффективности (то есть его параметры стремятся к истинным), и его оценки вероятностей состоятельны. Логистическая регрессия при этом тоже состоятельна, но может иметь большую дисперсию. Однако при нарушении нормальности LDA может давать смещённые оценки вероятностей, а логистическая регрессия остаётся робастной. Это классический пример bias‑variance trade‑off на уровне модели: структурные предположения уменьшают дисперсию, но добавляют смещение.

---

### 4. Диагностика предположений

Прежде чем довериться LDA или QDA, полезно проверить, насколько данные соответствуют нормальности внутри классов и одинаковы ли ковариационные матрицы.

**Проверка нормальности.** Построим Q‑Q графики и проведём тест Шапиро–Уилка для одного из признаков в каждом классе.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
feature_idx = 0   # alcohol
for ax, cls in zip(axes, np.unique(y_train)):
    class_data = X_train_s[y_train == cls, feature_idx]
    # Q‑Q plot вручную
    from scipy.stats import norm
    sorted_data = np.sort(class_data)
    theoretical_quantiles = norm.ppf(np.linspace(0.01, 0.99, len(class_data)))
    ax.scatter(theoretical_quantiles, sorted_data, s=10)
    ax.plot([-2.5, 2.5], [-2.5, 2.5], 'r--', alpha=0.5)
    ax.set_title(f'Класс {cls}, признак {wine.feature_names[feature_idx]}')
    ax.set_xlabel('Теоретические квантили'); ax.set_ylabel('Наблюдаемые квантили')
    stat, p = shapiro(class_data)
    print(f'Класс {cls}: Шапиро–Уилк p‑value = {p:.4f}')
plt.tight_layout()
plt.show()



*[Рисунок 4: Q‑Q графики для одного признака в трёх классах.]*

Если p‑value теста мало́, нормальность отвергается. Однако LDA часто остаётся хорошим линейным классификатором даже при умеренных отклонениях.

**Проверка равенства ковариационных матриц.** Визуализируем эллипсы рассеяния для первых двух признаков в каждом классе.




In [ ]:
from matplotlib.patches import Ellipse

def plot_cov_ellipse(mean, cov, ax, color, n_std=2.0):
    vals, vecs = la.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:,0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ellipse = Ellipse(xy=mean, width=width, height=height,
                      angle=angle, edgecolor=color, fc='None', lw=2, linestyle='--')
    ax.add_patch(ellipse)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['red', 'green', 'blue']
for cls, color in zip(np.unique(y_train), colors):
    Xk = X_train_s[y_train == cls][:, :2]
    mean_k = np.mean(Xk, axis=0)
    cov_k = np.cov(Xk.T)
    ax.scatter(Xk[:,0], Xk[:,1], alpha=0.5, color=color, label=f'Класс {cls}')
    plot_cov_ellipse(mean_k, cov_k, ax, color)
ax.legend()
ax.set_title('Эллипсы рассеяния (2σ) для первых двух признаков')
ax.set_xlabel(wine.feature_names[0])
ax.set_ylabel(wine.feature_names[1])
plt.tight_layout()
plt.show()



*[Рисунок 5: эллипсы рассеяния классов.]*

Если эллипсы заметно различаются по форме и ориентации, предположение LDA об общей ковариации нарушено — это аргумент в пользу QDA.

**Тест Бокса (Box’s M‑test).** Проверим гипотезу о равенстве ковариационных матриц формально. Реализуем упрощённую версию теста.




In [ ]:
def box_m_test(X, y):
    classes = np.unique(y)
    K = len(classes)
    n_total, d = X.shape
    groups = [X[y == k] for k in classes]
    n_k = np.array([len(g) for g in groups])

    # Pooled covariance
    pooled_cov = np.zeros((d, d))
    for i, g in enumerate(groups):
        pooled_cov += (n_k[i] - 1) * np.cov(g.T)
    pooled_cov /= (n_total - K)

    det_pooled = la.det(pooled_cov)
    M = 0.0
    for i, g in enumerate(groups):
        cov_i = np.cov(g.T)
        det_i = la.det(cov_i)
        if det_i <= 0:
            return np.nan, 0.0
        M += (n_k[i] - 1) * np.log(det_i)
    M = (n_total - K) * np.log(det_pooled) - M

    # поправка
    c = (2*d*d + 3*d - 1) / (6*(d+1)*(K-1)) * (np.sum(1/(n_k-1)) - 1/(n_total-K))
    df = d*(d+1)*(K-1)/2
    M_corrected = (1 - c) * M
    p_value = 1 - chi2.cdf(M_corrected, df)
    return M_corrected, p_value

M_stat, p_box = box_m_test(X_train_s, y_train)
print(f"Box's M‑test: M = {M_stat:.2f}, p‑value = {p_box:.4f}")



Если p‑value мало (например, < 0.05), нулевая гипотеза о равенстве матриц отвергается — QDA предпочтительнее. Но тест Бокса чувствителен к отклонениям от нормальности, поэтому его результаты следует интерпретировать с осторожностью.




In [ ]:
np.random.seed(42)
X0 = np.random.multivariate_normal([0,0], np.eye(2), 30)
X1 = np.random.multivariate_normal([2,2], np.eye(2), 30)
X_clean = np.vstack([X0, X1])
y_clean = np.array([0]*30 + [1]*30)

X_out = np.vstack([X_clean, [[10, -5]]])
y_out = np.append(y_clean, 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, X, y, title in zip(axes,
                           [X_clean, X_out],
                           [y_clean, y_out],
                           ['Без выброса', 'С выбросом (класс 0)']):
    lda = LinearDiscriminantAnalysis().fit(X, y)
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200),
                         np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
    Z = lda.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:,0], X[:,1], c=y, edgecolor='k', cmap='coolwarm', s=40)
    ax.set_title(title); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()


### Вопросы для самопроверки

#### Для начинающих
1. Как регуляризация помогает LDA? Что такое shrinkage?
2. В чём преимущество QDA перед LDA? Когда QDA может проигрывать?
3. Какую информацию даёт ROC‑кривая? Что такое AUC?
4. Почему LDA может быть лучше логистической регрессии при малых выборках?
5. Что показывает тест Бокса и как его результат влияет на выбор между LDA и QDA?

#### Для профессионалов
1. Выведите формулу для регуляризованной оценки ковариационной матрицы через выпуклую комбинацию. Как выбор параметра $\alpha$ влияет на смещение и дисперсию оценки?
2. Как обобщить LDA на случай, когда разные классы имеют разную стоимость ошибок? Запишите новое решающее правило.
3. Сравните асимптотическую эффективность LDA и логистической регрессии: когда предположение о нормальности верно, а когда — нет?
4. Опишите, как работает понижение размерности в LDA через поиск собственных векторов $\Sigma^{-1} \Sigma_B$ (solver='eigen') и через SVD (solver='svd'). В чём разница?
5. Предложите способ проверки предположения о нормальности, отличный от Q‑Q plot и теста Шапиро–Уилка, и обсудите его надёжность на малых выборках.



## Часть 3. LDA как инструмент визуализации, сравнение с другими методами и заключение

Мы построили классификаторы LDA и QDA, проверили их предположения. Теперь посмотрим на LDA с новой стороны — как на метод снижения размерности, который находит направления, максимально разделяющие классы. Затем сравним дискриминантный анализ с другими популярными классификаторами и подведём итоги.

---

### 1. LDA как метод снижения размерности (Fisher’s Linear Discriminant)

До сих пор мы использовали LDA для классификации. Однако сам Фишер в 1936 году предложил его для **снижения размерности** — поиска таких линейных комбинаций признаков, вдоль которых классы максимально разделены, а разброс внутри классов минимален. Это позволяет визуализировать многомерные данные на плоскости или подготовить их для других алгоритмов.

**Постановка задачи.** Пусть у нас $K$ классов. Ищем направление $\mathbf{w} \in \mathbb{R}^d$, на которое будем проецировать данные: $z = \mathbf{w}^\mathsf{T} \mathbf{x}$. Хотим, чтобы после проекции центры классов были как можно дальше друг от друга (большая межклассовая дисперсия) и одновременно чтобы точки одного класса группировались как можно теснее (малая внутриклассовая дисперсия). Формально максимизируем **критерий Фишера**:

$$
J(\mathbf{w}) = \frac{\mathbf{w}^\mathsf{T} S_B \mathbf{w}}{\mathbf{w}^\mathsf{T} S_W \mathbf{w}},
$$

где
- $S_W = \sum_{k=1}^K \sum_{i: y_i=k} (\mathbf{x}_i - \boldsymbol{\mu}_k)(\mathbf{x}_i - \boldsymbol{\mu}_k)^\mathsf{T}$ — **внутриклассовая матрица рассеяния** (within‑class scatter),
- $S_B = \sum_{k=1}^K n_k (\boldsymbol{\mu}_k - \boldsymbol{\mu})(\boldsymbol{\mu}_k - \boldsymbol{\mu})^\mathsf{T}$ — **межклассовая матрица рассеяния** (between‑class scatter),
- $\boldsymbol{\mu}_k$ — среднее класса $k$, $\boldsymbol{\mu}$ — общее среднее, $n_k$ — размер класса.

Чем больше $J(\mathbf{w})$, тем лучше разделение.

**Решение.** Дифференцируя $J(\mathbf{w})$ по $\mathbf{w}$ и приравнивая производную к нулю, получаем обобщённую задачу на собственные значения:

$$
S_B \mathbf{w} = \lambda S_W \mathbf{w}.
$$

Если $S_W$ обратима, это эквивалентно обычной задаче:

$$
S_W^{-1} S_B \mathbf{w} = \lambda \mathbf{w}.
$$

Наибольшее собственное значение соответствует наибольшему значению критерия Фишера, а соответствующий собственный вектор — искомому направлению. Для $K$ классов ранг $S_B$ не превышает $K-1$, поэтому можно извлечь до $K-1$ дискриминантных направлений (дискриминантных функций). Проекция на первые $r \le K-1$ из них даёт новое пространство пониженной размерности, оптимальное для разделения классов.

**Геометрический смысл** проще всего увидеть на примере двух классов. Представьте два вытянутых облака точек, почти соприкасающихся. Направление, соединяющее их центры, может быть не лучшим для разделения — если облака сильно вытянуты и пересекаются. LDA же находит направление, «перпендикулярное» эллипсам рассеяния, где перекрытие минимально (см. рисунок 6).


In [ ]:
np.random.seed(42)
mu0 = np.array([0., 0.])          # <-- float
mu1 = np.array([3., 1.])          # <-- float
Sigma = np.array([[2., 1.8], [1.8, 2.]])
X0 = np.random.multivariate_normal(mu0, Sigma, 200)
X1 = np.random.multivariate_normal(mu1, Sigma, 200)

S_W = (X0 - mu0).T @ (X0 - mu0) + (X1 - mu1).T @ (X1 - mu1)
w_lda = np.linalg.inv(S_W) @ (mu1 - mu0)
w_lda /= np.linalg.norm(w_lda)
w_cc = mu1 - mu0
w_cc /= np.linalg.norm(w_cc)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X0[:,0], X0[:,1], c='blue', alpha=0.4, s=20, label='Класс 0')
ax.scatter(X1[:,0], X1[:,1], c='red', alpha=0.4, s=20, label='Класс 1')

t = np.linspace(-4, 8, 100)
ax.plot(t * w_lda[0], t * w_lda[1], 'k-', linewidth=2, label='LDA')
ax.plot(t * w_cc[0], t * w_cc[1], 'gray', linestyle='--', linewidth=2,
        label='Центр–центр')

from matplotlib.patches import Ellipse

def plot_ellipse(mean, cov, ax, color):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*vecs[:,0][::-1]))
    w, h = 2 * np.sqrt(vals)
    ax.add_patch(Ellipse(mean, w, h, angle=angle, edgecolor=color,
                         fc='None', lw=2, linestyle='--'))

plot_ellipse(mu0, Sigma, ax, 'navy')
plot_ellipse(mu1, Sigma, ax, 'darkred')
ax.legend(); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Направление LDA vs направление центр–центр')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


*[Рисунок 6: иллюстрация одномерной проекции двух классов — PCA (пунктир) и LDA (сплошная линия). LDA даёт лучшее разделение.]*

**Отличие от PCA.** PCA ищет направления максимальной общей дисперсии, игнорируя метки классов. Если классы перекрываются вдоль направления наибольшего разброса, PCA может смешать их. LDA же сознательно максимизирует отношение межклассовой дисперсии к внутриклассовой, поэтому его проекции сохраняют разделимость. На практике PCA-проекция часто показывает общую структуру данных, а LDA‑проекция — структуру, связанную с классами.

> **Углублённый взгляд: связь с решением через собственные значения**
>
> Дифференцируя критерий $J(\mathbf{w}) = \frac{\mathbf{w}^\mathsf{T} S_B \mathbf{w}}{\mathbf{w}^\mathsf{T} S_W \mathbf{w}}$ по $\mathbf{w}$, имеем
> $$
> \frac{2 S_B \mathbf{w} (\mathbf{w}^\mathsf{T} S_W \mathbf{w}) - 2 S_W \mathbf{w} (\mathbf{w}^\mathsf{T} S_B \mathbf{w})}{(\mathbf{w}^\mathsf{T} S_W \mathbf{w})^2} = 0,
> $$
> откуда $S_B \mathbf{w} = \lambda S_W \mathbf{w}$ с $\lambda = \frac{\mathbf{w}^\mathsf{T} S_B \mathbf{w}}{\mathbf{w}^\mathsf{T} S_W \mathbf{w}} = J(\mathbf{w})$. Таким образом, каждая пара $(\lambda, \mathbf{w})$ — это обобщённая собственная пара матриц $(S_B, S_W)$. Поскольку $S_W$ симметрична и положительно определена, её можно разложить: $S_W = U \Lambda U^\mathsf{T}$. Переход к $\mathbf{v} = \Lambda^{1/2} U^\mathsf{T} \mathbf{w}$ сводит задачу к обычной симметричной проблеме $\tilde{S}_B \mathbf{v} = \lambda \mathbf{v}$, где $\tilde{S}_B = \Lambda^{-1/2} U^\mathsf{T} S_B U \Lambda^{-1/2}$. Наибольшему собственному значению $\lambda_1$ соответствует первая дискриминантная функция, и так далее.

> **Углублённый взгляд: LDA как частный случай канонического корреляционного анализа (CCA)**
>
> LDA можно рассматривать как CCA между матрицей признаков $\mathbf{X}$ и матрицей индикаторов классов $\mathbf{Y}$ (one‑hot encoding). Первая каноническая переменная CCA совпадает с первой дискриминантной функцией LDA. Эта связь открывает путь к обобщениям: например, регуляризованный LDA через регуляризованный CCA.

---

### 2. Практическое использование дискриминантных координат

Применим LDA как инструмент визуализации на наборах `iris` и `wine`, а также покажем, как предобработка снижением размерности влияет на точность последующих моделей.

#### Визуализация проекций (2 дискриминантные функции)




In [ ]:
# Импорты из предыдущих частей предполагаются выполненными
from sklearn.decomposition import PCA

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Стандартизация для PCA (LDA не так чувствителен, но для сравнения унифицируем)
scaler_iris = StandardScaler().fit(X_iris)
X_iris_s = scaler_iris.transform(X_iris)

# LDA — 2 компоненты
lda_iris = LinearDiscriminantAnalysis(n_components=2)
X_lda_iris = lda_iris.fit_transform(X_iris_s, y_iris)

# PCA — 2 компоненты
pca_iris = PCA(n_components=2)
X_pca_iris = pca_iris.fit_transform(X_iris_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in zip(axes,
                           [X_lda_iris, X_pca_iris],
                           ['LDA (2 дискр.)', 'PCA (2 комп.)']):
    scatter = ax.scatter(data[:, 0], data[:, 1], c=y_iris, cmap='viridis', edgecolor='k')
    ax.set_title(title)
    ax.set_xlabel('Первая компонента')
    ax.set_ylabel('Вторая компонента')
plt.colorbar(scatter, ax=axes, label='Класс')
plt.tight_layout()
plt.show()



*[Рисунок 7: проекции Iris на первые две компоненты LDA и PCA. LDA добивается почти полного разделения классов.]*

Видно, что LDA обеспечивает более компактные и разделённые группы, в то время как PCA смешивает классы 2 и 3 (зелёный и голубой).

На `wine`:




In [ ]:
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
scaler_wine = StandardScaler().fit(X_wine)
X_wine_s = scaler_wine.transform(X_wine)

lda_wine = LinearDiscriminantAnalysis(n_components=2)
X_lda_wine = lda_wine.fit_transform(X_wine_s, y_wine)

pca_wine = PCA(n_components=2)
X_pca_wine = pca_wine.fit_transform(X_wine_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in zip(axes,
                           [X_lda_wine, X_pca_wine],
                           ['LDA (2 дискр.)', 'PCA (2 комп.)']):
    ax.scatter(data[:, 0], data[:, 1], c=y_wine, cmap='plasma', edgecolor='k')
    ax.set_title(title)
    ax.set_xlabel('Первая компонента')
    ax.set_ylabel('Вторая компонента')
plt.tight_layout()
plt.show()



*[Рисунок 8: Wine: проекции LDA и PCA. LDA разделяет классы отчётливее.]*

#### LDA как предобработка перед классификацией

Снизим размерность до $K-1 = 2$ для Iris и обучим kNN и логистическую регрессию на проекциях. Сравним точность с обучением на всех 4 признаках.




In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_iris_s, y_iris,
                                                            test_size=0.3, random_state=42)

# Модели на полных признаках
knn_full = KNeighborsClassifier(n_neighbors=5).fit(X_train_i, y_train_i)
lr_full = LogisticRegression(max_iter=5000).fit(X_train_i, y_train_i)
acc_knn_full = accuracy_score(y_test_i, knn_full.predict(X_test_i))
acc_lr_full = accuracy_score(y_test_i, lr_full.predict(X_test_i))

# LDA-проекция (K-1 = 2)
lda_prep = LinearDiscriminantAnalysis(n_components=2).fit(X_train_i, y_train_i)
X_train_lda = lda_prep.transform(X_train_i)
X_test_lda = lda_prep.transform(X_test_i)

knn_lda = KNeighborsClassifier(n_neighbors=5).fit(X_train_lda, y_train_i)
lr_lda = LogisticRegression(max_iter=5000).fit(X_train_lda, y_train_i)
acc_knn_lda = accuracy_score(y_test_i, knn_lda.predict(X_test_lda))
acc_lr_lda = accuracy_score(y_test_i, lr_lda.predict(X_test_lda))

print(f"kNN на 4 признаках: {acc_knn_full:.4f}, на LDA-проекциях: {acc_knn_lda:.4f}")
print(f"Лог. регрессия на 4 признаках: {acc_lr_full:.4f}, на LDA-проекциях: {acc_lr_lda:.4f}")



Результаты показывают, что точность на проекциях часто не уступает полной размерности, а иногда даже выше за счёт устранения шума. Это делает LDA полезным этапом пайплайна.

---

### 3. Сравнение с другими классификаторами

Дискриминантный анализ занимает особое место среди классификаторов. Сравним его с наиболее близкими методами.

| Критерий | LDA/QDA | Логистическая регрессия | kNN | SVM (линейное ядро) |
|----------|---------|--------------------------|-----|----------------------|
| **Тип границы** | Линейная (LDA), квадратичная (QDA) | Линейная (без полиномов) | Произвольная (кусочно‑линейная) | Линейная (или нелинейная с ядрами) |
| **Вероятностный выход** | Да (хорошо калиброван, если верна нормальность) | Да (хорошо калиброван) | Оценка долей соседей (грубая) | Нет (только через Платта) |
| **Число параметров** | LDA: $Kd + \frac{d(d+1)}{2}$, QDA: больше | $d+1$ (на класс) | Нет явных параметров | Зависит от числа опорных векторов |
| **Чувствительность к выбросам** | Высокая (средние и ковариация) | Умеренная | Высокая при малых $k$ | Высокая (опорные векторы у границы) |
| **Интерпретируемость** | Высокая (веса LDA имеют смысл) | Высокая (коэффициенты регрессии) | Низкая | Средняя (направляющий вектор) |
| **Требования к данным** | Нормальность, равенство ковариаций (LDA) | Нет | Нет (метрика, масштаб) | Нет (масштаб важен) |

**LDA vs логистическая регрессия** — классический спор генеративного и дискриминативного подходов. Если предположение о нормальности выполняется, LDA асимптотически эффективен и может давать лучшие результаты на малых выборках. Логистическая регрессия более робастна к отклонениям от нормальности.

**LDA vs kNN**: kNN не делает предположений о форме классов, но страдает от проклятия размерности и требует хранения всей выборки. LDA даёт компактную модель с линейной границей, быструю на этапе предсказания, но может проигрывать, если истинная граница сильно нелинейна.

**LDA vs линейное SVM**: SVM максимизирует зазор, что делает его устойчивее к выбросам вблизи границы. LDA опирается на средние и ковариации, поэтому одиночные далёкие выбросы могут сильно сместить границу. Однако LDA предоставляет вероятности классов напрямую, а SVM — нет.

---

### 4. Заключение и резюме главы

Линейный и квадратичный дискриминантный анализ — это воплощение генеративного подхода к классификации: мы моделируем, как признаки порождаются каждым классом, и с помощью теоремы Байеса получаем вероятности принадлежности. Ключевые выводы:

- **Байесовский классификатор** оптимален при известных распределениях. Предположив нормальность, мы получаем LDA и QDA.
- **LDA** предполагает общую ковариационную матрицу и даёт линейные границы; **QDA** разрешает каждому классу свою матрицу и даёт квадратичные границы.
- LDA служит не только классификатором, но и **мощным методом снижения размерности**, максимизируя разделимость классов (критерий Фишера).
- Компромисс смещения и дисперсии: LDA более смещён, но стабилен; QDA гибче, но требует больше данных.
- Генеративные классификаторы дают калиброванные вероятности и могут быть эффективнее дискриминативных при верной спецификации модели.

**Итоговая таблица LDA/QDA:**

| Свойство | LDA | QDA |
|----------|-----|-----|
| Предположение о данных | $\mathcal{N}(\boldsymbol{\mu}_k, \Sigma)$ | $\mathcal{N}(\boldsymbol{\mu}_k, \Sigma_k)$ |
| Дискриминантная функция | Линейная: $\mathbf{x}^\mathsf{T}\Sigma^{-1}\boldsymbol{\mu}_k - \frac12 \boldsymbol{\mu}_k^\mathsf{T}\Sigma^{-1}\boldsymbol{\mu}_k + \log\pi_k$ | Квадратичная: $-\frac12 \log|\Sigma_k| - \frac12 (\mathbf{x}-\boldsymbol{\mu}_k)^\mathsf{T}\Sigma_k^{-1}(\mathbf{x}-\boldsymbol{\mu}_k) + \log\pi_k$ |
| Граница | Гиперплоскость | Квадрика (эллипс, гипербола и т.д.) |
| Число параметров | $Kd + \frac{d(d+1)}{2}$ | $K\left(d + \frac{d(d+1)}{2}\right)$ |
| Рекомендация | Мало данных, нужна интерпретация, визуализация | Много данных, ковариации классов различаются |

**Практические рекомендации:**
- Если данных мало ($n < 5d$), начинайте с LDA, при коллинеарности — с shrinkage.
- Если данные примерно нормальны и ковариации классов схожи, LDA будет почти оптимален.
- Если есть основания полагать, что разброс классов разный, и данных достаточно, попробуйте QDA.
- Для визуализации многомерных данных с метками всегда используйте LDA‑проекции наряду с PCA — они расскажут о структуре классов.

---

### Вопросы для самопроверки

#### Для начинающих
1. Какую величину максимизирует критерий Фишера в LDA? Что она измеряет?
2. Сколько дискриминантных направлений можно извлечь для трёх классов?
3. Чем LDA-проекция отличается от PCA-проекции? Почему LDA лучше разделяет классы?
4. Когда следует предпочесть QDA вместо LDA?
5. Что даёт shrinkage в LDA?

#### Для профессионалов
1. Докажите, что решение $S_W^{-1} S_B \mathbf{w} = \lambda \mathbf{w}$ эквивалентно максимизации критерия Фишера.
2. Как собственные значения $\lambda$ связаны с разделимостью классов? Что означает $\lambda \gg 1$ и $\lambda \approx 0$?
3. Почему LDA-проекции не ортогональны в исходном пространстве, но некоррелированы в смысле внутриклассовой дисперсии?
4. Сравните асимптотическую ошибку LDA и наивного байесовского классификатора в случае, когда признаки коррелированы.
5. Обобщите критерий Фишера на случай регуляризованного LDA (shrinkage), записав новую целевую функцию.

---

### Задания повышенной сложности

1. **Аналитическое решение для бинарного LDA.** Для двух классов с равными априорными вероятностями и общей ковариационной матрицей $\Sigma$ выведите, что разделяющая гиперплоскость задаётся вектором $\mathbf{w} = \Sigma^{-1}(\boldsymbol{\mu}_1 - \boldsymbol{\mu}_2)$, и покажите, что она перпендикулярна направлению, соединяющему центры в смысле расстояния Махаланобиса. Проверьте, что это совпадает с первой дискриминантной функцией Фишера.

2. **Реализация LDA через обобщённую задачу на собственные значения.** Не используя `scipy.linalg.eigh`, реализуйте метод `fit_transform` для LDA вручную: вычислите $S_W$ и $S_B$, решите обобщённую проблему через разложение Холецкого и симметричную проблему. Сравните результат с `sklearn`.

3. **Эмпирическое сравнение на смесях распределений.** Сгенерируйте данные, где признаки внутри классов распределены как смесь двух нормальных распределений (нарушение нормальности). Для разных объёмов выборки $n$ постройте кривые ошибки на тесте для LDA, QDA и логистической регрессии. Оцените, насколько LDA/QDA чувствительны к форме распределения.

4. **Эквивалентность LDA и линейной регрессии (доказательство).** В случае бинарной классификации с равными размерами классов докажите, что коэффициенты LDA пропорциональны коэффициентам линейной регрессии, предсказывающей индикатор класса $y \in \{0,1\}$. Указание: выразите $\hat{\boldsymbol{\beta}}_{LS}$ через средние и матрицу $X^\mathsf{T} X$, а затем покажите, что $\Sigma^{-1}(\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0)$ пропорционален этому вектору при сбалансированных классах.